# Adaptive Optics Wavefront Sensing with Shack-Hartmann Sensor

## Problem Background and Motivation

Adaptive optics (AO) is a technology used to improve the performance of optical systems by reducing the effects of wavefront distortions. Originally developed for astronomical telescopes to compensate for atmospheric turbulence, AO systems are now widely used in laser communications, ophthalmology, and microscopy.

### The Inverse Problem in Adaptive Optics

The fundamental challenge in AO is an **inverse problem**: given measurements of a distorted wavefront, we must reconstruct the wavefront aberrations and compute the appropriate correction commands for a deformable mirror (DM). This is inherently ill-posed because:

1. **Incomplete information**: The Shack-Hartmann wavefront sensor (SH-WFS) measures local wavefront slopes, not the wavefront itself
2. **Noise**: Photon noise and detector noise corrupt the measurements
3. **Temporal dynamics**: The atmosphere evolves faster than our correction bandwidth
4. **Spatial aliasing**: High-frequency aberrations can alias into low-frequency measurements

### The Shack-Hartmann Wavefront Sensor

The SH-WFS, invented by Roland Shack and Ben Platt in 1971 (building on Johannes Hartmann's 1900 screen test), divides the telescope pupil into subapertures using a lenslet array. Each lenslet forms a spot on a detector, and the displacement of each spot from its reference position is proportional to the local wavefront slope.

### Closed-Loop Control

In a closed-loop AO system, we continuously:
1. **Measure** the residual wavefront error using the WFS
2. **Reconstruct** the wavefront from slope measurements
3. **Command** the DM to apply a correction
4. **Repeat** at high temporal frequency (typically 500-2000 Hz)

This tutorial implements a complete AO simulation using the Object-Oriented Python Adaptive Optics (OOPAO) framework, demonstrating the inverse problem of wavefront reconstruction and closed-loop correction.

**Key Reference**: Roddier, F. (1999). *Adaptive Optics in Astronomy*. Cambridge University Press.

## Mathematical Formulation

### The Forward Model: From Wavefront to Slopes

The atmospheric turbulence introduces a phase aberration $\phi(\mathbf{r})$ across the telescope pupil. The Shack-Hartmann WFS measures the local wavefront slopes (gradients) in each subaperture:

$$
s_x^{(i)} = \frac{1}{A_i} \iint_{\Omega_i} \frac{\partial \phi(\mathbf{r})}{\partial x} \, d^2\mathbf{r} \quad \text{(Eq. 1)}
$$

$$
s_y^{(i)} = \frac{1}{A_i} \iint_{\Omega_i} \frac{\partial \phi(\mathbf{r})}{\partial y} \, d^2\mathbf{r} \quad \text{(Eq. 2)}
$$

where $\Omega_i$ is the $i$-th subaperture region and $A_i$ is its area.

### Modal Decomposition

We decompose the DM shape using a modal basis (Karhunen-Loève modes optimized for atmospheric turbulence):

$$
\phi_{\text{DM}}(\mathbf{r}) = \sum_{j=1}^{N_m} a_j \, M_j(\mathbf{r}) = \mathbf{M} \cdot \mathbf{a} \quad \text{(Eq. 3)}
$$

where $M_j(\mathbf{r})$ are the modal basis functions and $a_j$ are the modal coefficients.

### The Interaction Matrix

The relationship between modal coefficients and measured slopes is linear and captured by the **interaction matrix** $\mathbf{D}$:

$$
\mathbf{s} = \mathbf{D} \cdot \mathbf{a} + \mathbf{n} \quad \text{(Eq. 4)}
$$

where $\mathbf{s} \in \mathbb{R}^{2N_s}$ is the slope vector (x and y slopes for $N_s$ valid subapertures), $\mathbf{a} \in \mathbb{R}^{N_m}$ is the modal coefficient vector, and $\mathbf{n}$ is measurement noise.

### The Inverse Problem: Wavefront Reconstruction

Given measured slopes $\mathbf{s}$, we seek to estimate the modal coefficients $\hat{\mathbf{a}}$ by solving:

$$
\hat{\mathbf{a}} = \arg\min_{\mathbf{a}} \|\mathbf{D} \mathbf{a} - \mathbf{s}\|_2^2 \quad \text{(Eq. 5)}
$$

The least-squares solution uses the **reconstructor matrix** $\mathbf{R}$:

$$
\mathbf{R} = (\mathbf{D}^T \mathbf{D})^{-1} \mathbf{D}^T = \mathbf{V} \mathbf{\Sigma}^{-1} \mathbf{U}^T \quad \text{(Eq. 6)}
$$

where we use SVD: $\mathbf{D} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T$. Regularization is applied by truncating small singular values.

### Closed-Loop Integral Control

The DM commands are updated using an integral controller:

$$
\mathbf{u}^{(k+1)} = \mathbf{u}^{(k)} - g \cdot \mathbf{R}_{\text{zonal}} \cdot \mathbf{s}^{(k)} \quad \text{(Eq. 7)}
$$

where $g \in (0, 1)$ is the loop gain, $\mathbf{u}^{(k)}$ are the actuator commands at iteration $k$, and $\mathbf{R}_{\text{zonal}} = \mathbf{M} \cdot \mathbf{R}_{\text{modal}}$ converts modal reconstruction to actuator commands.

### Performance Metric: Strehl Ratio

The Strehl ratio quantifies AO performance as the ratio of peak intensity to diffraction-limited peak:

$$
\text{SR} = \frac{\max(\text{PSF}_{\text{corrected}})}{\max(\text{PSF}_{\text{perfect}})} \approx \exp\left(-\sigma_\phi^2\right) \quad \text{(Eq. 8)}
$$

where $\sigma_\phi^2$ is the residual wavefront variance in radians squared (Maréchal approximation).

In [ ]:
# =============================================================================
# Cell 3: Environment Setup & Imports
# =============================================================================

import sys
import os
import time
import warnings
warnings.filterwarnings('ignore')

# Add OOPAO to path
sys.path.append('/home/yjh/OOPAO')

# Core scientific computing
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

# OOPAO components for adaptive optics simulation
from OOPAO.Telescope import Telescope
from OOPAO.Source import Source
from OOPAO.Atmosphere import Atmosphere
from OOPAO.DeformableMirror import DeformableMirror
from OOPAO.ShackHartmann import ShackHartmann
from OOPAO.Detector import Detector
from OOPAO.calibration.compute_KL_modal_basis import compute_KL_basis

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib for publication-quality figures
plt.rcParams.update({
    'figure.figsize': (12, 4),
    'figure.dpi': 120,
    'font.size': 11,
    'font.family': 'serif',
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'image.cmap': 'viridis',
    'axes.grid': True,
    'grid.alpha': 0.3
})

# Print library versions
print("="*60)
print("Environment Setup Complete")
print("="*60)
print(f"NumPy version: {np.__version__}")
print(f"Python version: {sys.version.split()[0]}")
print(f"OOPAO path: /home/yjh/OOPAO")
print("="*60)

## Forward Model Explanation

### Physical Process of Shack-Hartmann Wavefront Sensing

The forward model in our AO system describes how light propagates from a distant star through the turbulent atmosphere, telescope, deformable mirror, and finally to the wavefront sensor and science camera.

#### 1. Atmospheric Turbulence

The atmosphere introduces a random phase screen $\phi_{\text{atm}}(\mathbf{r}, t)$ that evolves according to the frozen-flow hypothesis (Taylor's hypothesis). The turbulence strength is characterized by the Fried parameter $r_0$, which represents the aperture diameter over which the RMS wavefront error is approximately 1 radian.

#### 2. Telescope and Deformable Mirror

The telescope collects light and the DM applies a corrective phase:

$$
\phi_{\text{residual}}(\mathbf{r}) = \phi_{\text{atm}}(\mathbf{r}) - \phi_{\text{DM}}(\mathbf{r})
$$

#### 3. Shack-Hartmann Measurement Process

Each lenslet in the SH-WFS samples a portion of the pupil. The electric field at each lenslet is:

$$
E_i(\mathbf{r}) = A(\mathbf{r}) \cdot \exp\left(i \phi_{\text{residual}}(\mathbf{r})\right) \cdot P_i(\mathbf{r})
$$

where $A(\mathbf{r})$ is the amplitude and $P_i(\mathbf{r})$ is the $i$-th lenslet pupil function.

The spot intensity on the detector is formed by Fourier optics:

$$
I_i(\mathbf{u}) = \left| \mathcal{F}\{E_i(\mathbf{r})\} \right|^2
$$

#### 4. Centroid Calculation

The spot displacement is measured using center-of-gravity (CoG) centroiding:

$$
c_x^{(i)} = \frac{\sum_{p,q} x_p \cdot I_i(x_p, y_q)}{\sum_{p,q} I_i(x_p, y_q)}, \quad c_y^{(i)} = \frac{\sum_{p,q} y_q \cdot I_i(x_p, y_q)}{\sum_{p,q} I_i(x_p, y_q)}
$$

The slopes are then: $s_x^{(i)} = c_x^{(i)} - c_{x,\text{ref}}^{(i)}$ (and similarly for $y$).

In [ ]:
# =============================================================================
# Cell 5: Forward Model & System Initialization
# =============================================================================

def initialize_ao_system(n_subaperture=20, n_modes=20):
    """
    Initialize all components of the adaptive optics system.
    
    Parameters:
    -----------
    n_subaperture : int
        Number of subapertures across the pupil diameter
    n_modes : int
        Number of KL modes for wavefront reconstruction
    
    Returns:
    --------
    dict : Dictionary containing all system components
    """
    print("Initializing AO System Components...")
    print("-" * 50)
    
    # Telescope: 8m diameter (similar to VLT)
    n_pix_pupil = 6 * n_subaperture  # 6 pixels per subaperture
    tel = Telescope(
        resolution=n_pix_pupil,
        diameter=8.0,  # meters
        samplingTime=1/1000,  # 1 kHz sampling
        centralObstruction=0.0  # No central obstruction
    )
    print(f"  Telescope: D={tel.D}m, resolution={tel.resolution}px")
    
    # Natural Guide Star at infinity
    ngs = Source(optBand='I', magnitude=8, coordinates=[0, 0])
    ngs * tel  # Couple source to telescope
    print(f"  Source: {ngs.optBand}-band, magnitude={ngs.magnitude}")
    
    # Atmospheric turbulence: single layer
    atm = Atmosphere(
        telescope=tel,
        r0=0.15,  # Fried parameter at 500nm
        L0=25,    # Outer scale
        fractionalR0=[1.0],
        windSpeed=[10],  # m/s
        windDirection=[0],
        altitude=[0]  # Ground layer
    )
    atm.initializeAtmosphere(tel)
    print(f"  Atmosphere: r0={atm.r0}m, L0={atm.L0}m, wind={atm.windSpeed[0]}m/s")
    
    # Deformable Mirror
    dm = DeformableMirror(
        telescope=tel,
        nSubap=n_subaperture,
        mechCoupling=0.35  # Actuator coupling coefficient
    )
    print(f"  DM: {dm.nValidAct} valid actuators")
    
    # Shack-Hartmann Wavefront Sensor
    wfs = ShackHartmann(
        nSubap=n_subaperture,
        telescope=tel,
        lightRatio=0.5  # Minimum flux ratio for valid subaperture
    )
    print(f"  WFS: {n_subaperture}x{n_subaperture} subapertures, {wfs.nValidSubaperture} valid")
    
    # Science camera (high resolution for PSF/Strehl)
    sci_cam = Detector(tel.resolution * 2)
    print(f"  Science camera: {sci_cam.resolution}x{sci_cam.resolution} pixels")
    
    print("-" * 50)
    print("System initialization complete!\n")
    
    return {
        'tel': tel,
        'ngs': ngs,
        'atm': atm,
        'dm': dm,
        'wfs': wfs,
        'sci_cam': sci_cam,
        'n_subaperture': n_subaperture,
        'n_modes': n_modes
    }


def get_slopes_diffractive(wfs, tel, ngs):
    """
    Compute WFS slopes using diffractive (FFT-based) spot formation.
    This simulates the physical process of the Shack-Hartmann sensor.
    
    Parameters:
    -----------
    wfs : ShackHartmann
        The wavefront sensor object
    tel : Telescope
        The telescope object (contains current phase)
    ngs : Source
        The guide star source
    
    Returns:
    --------
    slopes : ndarray
        Concatenated x and y slopes [sx1, sx2, ..., sy1, sy2, ...]
    """
    # Get electric field at each lenslet
    cube_em = wfs.get_lenslet_em_field(tel.src.phase)
    
    # Form spots via FFT (Fraunhofer diffraction)
    complex_field = np.fft.fft2(cube_em, axes=[1, 2])
    intensity_spots = np.abs(complex_field) ** 2
    
    # Center-of-gravity centroiding
    n_pix = intensity_spots.shape[1]
    x = np.arange(n_pix) - n_pix // 2
    X, Y = np.meshgrid(x, x)
    
    slopes = np.zeros((wfs.nValidSubaperture, 2))
    valid_idx = 0
    
    for i in range(wfs.nSubap ** 2):
        if wfs.valid_subapertures_1D[i]:
            I = intensity_spots[i]
            flux = np.sum(I)
            if flux > 0:
                cx = np.sum(I * X) / flux
                cy = np.sum(I * Y) / flux
                slopes[valid_idx, 0] = cx
                slopes[valid_idx, 1] = cy
                valid_idx += 1
    
    # Return as flat vector [sx, sy]
    return np.concatenate((slopes[:, 0], slopes[:, 1]))


def compute_reference_data(system):
    """
    Compute reference PSF (diffraction-limited) and reference slopes (flat wavefront).
    """
    tel = system['tel']
    ngs = system['ngs']
    wfs = system['wfs']
    sci_cam = system['sci_cam']
    
    # Reset telescope OPD to zero (perfect wavefront)
    tel.resetOPD()
    
    # Compute reference PSF
    ngs * tel * sci_cam
    psf_ref = sci_cam.frame.copy()
    
    # Compute reference slopes
    ref_slopes = get_slopes_diffractive(wfs, tel, ngs)
    
    return psf_ref, ref_slopes


# Initialize the system
print("="*60)
print("FORWARD MODEL: AO System Initialization")
print("="*60 + "\n")

system = initialize_ao_system(n_subaperture=20, n_modes=20)
psf_ref, ref_slopes = compute_reference_data(system)

print(f"Reference PSF shape: {psf_ref.shape}")
print(f"Reference PSF peak: {np.max(psf_ref):.4e}")
print(f"Reference slopes shape: {ref_slopes.shape}")
print(f"Number of slope measurements: {len(ref_slopes)}")

# Visualize reference PSF
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Linear scale
im0 = axes[0].imshow(psf_ref, cmap='hot', origin='lower')
axes[0].set_title('Reference PSF (Linear Scale)')
axes[0].set_xlabel('Pixels')
axes[0].set_ylabel('Pixels')
plt.colorbar(im0, ax=axes[0], label='Intensity')

# Log scale
im1 = axes[1].imshow(np.log10(psf_ref + 1e-10), cmap='hot', origin='lower')
axes[1].set_title('Reference PSF (Log Scale)')
axes[1].set_xlabel('Pixels')
axes[1].set_ylabel('Pixels')
plt.colorbar(im1, ax=axes[1], label='log₁₀(Intensity)')

plt.tight_layout()
plt.show()

## Reconstruction Algorithm: Calibration and Closed-Loop Control

### Step 1: Interaction Matrix Calibration

The interaction matrix $\mathbf{D}$ is measured experimentally using a **push-pull** technique:

1. For each mode $j$, apply a positive poke: $a_j = +\delta$
2. Measure slopes: $\mathbf{s}^+$
3. Apply negative poke: $a_j = -\delta$
4. Measure slopes: $\mathbf{s}^-$
5. Compute column $j$ of $\mathbf{D}$: $\mathbf{D}_{:,j} = \frac{\mathbf{s}^+ - \mathbf{s}^-}{2\delta}$

This push-pull approach cancels static aberrations and reduces noise.

### Step 2: Reconstructor Computation via SVD

The reconstructor is computed using truncated SVD for regularization:

$$
\mathbf{D} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T
$$

$$
\mathbf{R}_{\text{modal}} = \mathbf{V} \mathbf{\Sigma}^{-1}_{\text{reg}} \mathbf{U}^T
$$

where $\mathbf{\Sigma}^{-1}_{\text{reg}}$ has entries:
$$
\sigma_i^{-1} = \begin{cases} 1/\sigma_i & \text{if } \sigma_i > \epsilon \cdot \sigma_{\max} \\ 0 & \text{otherwise} \end{cases}
$$

### Step 3: Zonal Reconstructor

Convert from modal to actuator commands:
$$
\mathbf{R}_{\text{zonal}} = \mathbf{M}_{\text{modes}} \cdot \mathbf{R}_{\text{modal}}
$$

### Step 4: Integral Control Law

The closed-loop controller uses integral feedback:
$$
\mathbf{u}^{(k+1)} = \mathbf{u}^{(k)} - g \cdot \mathbf{R}_{\text{zonal}} \cdot (\mathbf{s}^{(k)} - \mathbf{s}_{\text{ref}})
$$

**Convergence Properties:**
- The loop is stable for $0 < g < 2/\lambda_{\max}$ where $\lambda_{\max}$ is the largest eigenvalue of the system
- Typical gains: $g \in [0.3, 0.6]$ balance speed vs. stability
- Higher gains converge faster but may oscillate
- Lower gains are more robust to noise but slower

**Regularization Strategy:**
- SVD truncation removes poorly-sensed modes
- Threshold typically set at $10^{-3}$ to $10^{-2}$ of maximum singular value
- This prevents noise amplification in the reconstruction

In [ ]:
# =============================================================================
# Cell 7: Reconstruction Implementation - Calibration & Closed-Loop
# =============================================================================

def calibrate_system(system, ref_slopes, stroke=1e-8):
    """
    Calibrate the AO system by computing the interaction matrix and reconstructor.
    
    Parameters:
    -----------
    system : dict
        System components dictionary
    ref_slopes : ndarray
        Reference slopes for flat wavefront
    stroke : float
        Amplitude for push-pull calibration (meters)
    
    Returns:
    --------
    dict : Calibration data including interaction matrix and reconstructor
    """
    tel = system['tel']
    ngs = system['ngs']
    atm = system['atm']
    dm = system['dm']
    wfs = system['wfs']
    n_modes = system['n_modes']
    
    print("Calibrating AO System...")
    print("-" * 50)
    
    # Compute KL modal basis (optimized for atmospheric turbulence)
    print("  Computing KL modal basis...")
    M2C_KL = compute_KL_basis(tel, atm, dm, lim=0)
    basis_modes = M2C_KL[:, :n_modes]
    print(f"  Modal basis shape: {basis_modes.shape}")
    
    # Build Interaction Matrix via Push-Pull
    print(f"  Building interaction matrix ({n_modes} modes)...")
    n_meas = wfs.nSignal
    interaction_matrix = np.zeros((n_meas, n_modes))
    
    tel.resetOPD()
    
    for i in range(n_modes):
        # Push: positive mode amplitude
        dm.coefs = basis_modes[:, i] * stroke
        ngs * tel * dm
        slopes_push = get_slopes_diffractive(wfs, tel, ngs)
        
        # Pull: negative mode amplitude
        dm.coefs = -basis_modes[:, i] * stroke
        ngs * tel * dm
        slopes_pull = get_slopes_diffractive(wfs, tel, ngs)
        
        # Interaction matrix column
        interaction_matrix[:, i] = (slopes_push - slopes_pull) / (2 * stroke)
    
    dm.coefs[:] = 0  # Reset DM
    print(f"  Interaction matrix shape: {interaction_matrix.shape}")
    
    # Compute Reconstructor via SVD with regularization
    print("  Computing reconstructor via SVD...")
    U, s, Vt = np.linalg.svd(interaction_matrix, full_matrices=False)
    
    # Regularization: truncate small singular values
    threshold = 1e-3
    s_inv = np.zeros_like(s)
    n_valid = np.sum(s > threshold * s[0])
    s_inv[s > threshold * s[0]] = 1.0 / s[s > threshold * s[0]]
    print(f"  Singular values: max={s[0]:.4e}, min={s[-1]:.4e}")
    print(f"  Valid modes after truncation: {n_valid}/{n_modes}")
    
    # Modal reconstructor
    reconstructor_modal = Vt.T @ np.diag(s_inv) @ U.T
    
    # Convert to zonal (actuator) reconstructor
    reconstructor_zonal = basis_modes @ reconstructor_modal
    print(f"  Zonal reconstructor shape: {reconstructor_zonal.shape}")
    
    print("-" * 50)
    print("Calibration complete!\n")
    
    return {
        'interaction_matrix': interaction_matrix,
        'reconstructor': reconstructor_zonal,
        'basis_modes': basis_modes,
        'singular_values': s,
        'n_valid_modes': n_valid
    }


def run_closed_loop(system, calibration, ref_slopes, psf_ref, n_iter=20, gain=0.4):
    """
    Run closed-loop AO correction using integral control.
    
    Parameters:
    -----------
    system : dict
        System components
    calibration : dict
        Calibration data (reconstructor, etc.)
    ref_slopes : ndarray
        Reference slopes
    psf_ref : ndarray
        Reference PSF for Strehl calculation
    n_iter : int
        Number of iterations
    gain : float
        Loop gain (0 < gain < 1)
    
    Returns:
    --------
    dict : Results including Strehl history, final PSF, etc.
    """
    tel = system['tel']
    ngs = system['ngs']
    atm = system['atm']
    dm = system['dm']
    wfs = system['wfs']
    sci_cam = system['sci_cam']
    reconstructor = calibration['reconstructor']
    
    print(f"Running Closed-Loop AO ({n_iter} iterations, gain={gain})")
    print("-" * 50)
    
    # Initialize
    dm.coefs[:] = 0
    strehl_history = []
    residual_slopes_history = []
    dm_rms_history = []
    
    for k in range(n_iter):
        # Update atmosphere (frozen flow)
        atm.update()
        
        # Forward pass: atmosphere -> telescope -> DM
        atm * ngs * tel * dm
        
        # Measure slopes (subtract reference)
        slopes_meas = get_slopes_diffractive(wfs, tel, ngs) - ref_slopes
        residual_slopes_history.append(np.sqrt(np.mean(slopes_meas**2)))
        
        # Integral control: u[k+1] = u[k] - gain * R * s[k]
        delta_command = np.matmul(reconstructor, slopes_meas)
        dm.coefs = dm.coefs - gain * delta_command
        dm_rms_history.append(np.sqrt(np.mean(dm.coefs**2)))
        
        # Science path: compute PSF and Strehl
        atm * ngs * tel * dm * sci_cam
        sr = compute_strehl(sci_cam.frame, psf_ref)
        strehl_history.append(sr)
        
        print(f"  Iter {k+1:02d}: Strehl = {sr:6.2f}%, Slope RMS = {residual_slopes_history[-1]:.4e}")
    
    print("-" * 50)
    print("Closed-loop complete!\n")
    
    return {
        'strehl_history': np.array(strehl_history),
        'residual_slopes_history': np.array(residual_slopes_history),
        'dm_rms_history': np.array(dm_rms_history),
        'final_psf': sci_cam.frame.copy(),
        'final_dm_commands': dm.coefs.copy()
    }


def compute_strehl(psf, psf_ref):
    """
    Compute Strehl ratio using OTF method.
    Strehl = Sum(OTF) / Sum(OTF_perfect)
    """
    otf = np.abs(np.fft.fftshift(np.fft.fft2(psf)))
    otf_ref = np.abs(np.fft.fftshift(np.fft.fft2(psf_ref)))
    strehl = np.sum(otf) / np.sum(otf_ref)
    return strehl * 100  # Percent


# Run calibration
print("="*60)
print("CALIBRATION PHASE")
print("="*60 + "\n")

calibration = calibrate_system(system, ref_slopes, stroke=1e-8)

# Run closed-loop
print("="*60)
print("CLOSED-LOOP CORRECTION")
print("="*60 + "\n")

results = run_closed_loop(
    system, calibration, ref_slopes, psf_ref,
    n_iter=20, gain=0.4
)

## Results Visualization

We will now visualize the results of our closed-loop AO correction. The key visualizations include:

1. **PSF Comparison**: Side-by-side comparison of the open-loop (uncorrected), closed-loop (corrected), and reference (diffraction-limited) PSFs. Look for:
   - Concentration of energy in the core
   - Reduction of the halo/speckle pattern
   - Approach toward the Airy pattern

2. **Strehl Ratio Evolution**: The Strehl ratio should increase over iterations as the loop converges. Typical behavior:
   - Initial rapid improvement (first few iterations)
   - Gradual convergence to steady-state
   - Possible oscillations if gain is too high

3. **DM Shape**: The final DM surface shows the correction applied. It should be the negative of the atmospheric aberration (within the correctable spatial frequencies).

**Expected Performance:**
- For $D/r_0 \approx 53$ (8m telescope, $r_0 = 0.15$m), uncorrected Strehl is very low (<1%)
- With 20 modes corrected, we expect Strehl of 20-50% depending on conditions
- Higher mode counts and faster loops would achieve higher Strehl

In [ ]:
# =============================================================================
# Cell 9: Visualization - Ground Truth vs Reconstruction
# =============================================================================

# Get an open-loop PSF for comparison
system['dm'].coefs[:] = 0
system['atm'].update()
system['atm'] * system['ngs'] * system['tel'] * system['sci_cam']
psf_open_loop = system['sci_cam'].frame.copy()
strehl_open = compute_strehl(psf_open_loop, psf_ref)

# Create comprehensive visualization
fig = plt.figure(figsize=(16, 10))

# Row 1: PSF comparisons
ax1 = fig.add_subplot(2, 3, 1)
im1 = ax1.imshow(np.log10(psf_open_loop + 1e-10), cmap='hot', origin='lower',
                  vmin=-6, vmax=0)
ax1.set_title(f'Open-Loop PSF\nStrehl = {strehl_open:.2f}%')
ax1.set_xlabel('Pixels')
ax1.set_ylabel('Pixels')
plt.colorbar(im1, ax=ax1, label='log₁₀(I)', shrink=0.8)

ax2 = fig.add_subplot(2, 3, 2)
im2 = ax2.imshow(np.log10(results['final_psf'] + 1e-10), cmap='hot', origin='lower',
                  vmin=-6, vmax=0)
ax2.set_title(f'Closed-Loop PSF\nStrehl = {results["strehl_history"][-1]:.2f}%')
ax2.set_xlabel('Pixels')
ax2.set_ylabel('Pixels')
plt.colorbar(im2, ax=ax2, label='log₁₀(I)', shrink=0.8)

ax3 = fig.add_subplot(2, 3, 3)
im3 = ax3.imshow(np.log10(psf_ref + 1e-10), cmap='hot', origin='lower',
                  vmin=-6, vmax=0)
ax3.set_title('Reference PSF\n(Diffraction Limited)')
ax3.set_xlabel('Pixels')
ax3.set_ylabel('Pixels')
plt.colorbar(im3, ax=ax3, label='log₁₀(I)', shrink=0.8)

# Row 2: PSF cross-sections and DM shape
ax4 = fig.add_subplot(2, 3, 4)
center = psf_ref.shape[0] // 2
ax4.semilogy(psf_open_loop[center, :], 'r-', label='Open-Loop', alpha=0.7)
ax4.semilogy(results['final_psf'][center, :], 'b-', label='Closed-Loop', linewidth=2)
ax4.semilogy(psf_ref[center, :], 'k--', label='Reference', linewidth=1.5)
ax4.set_xlabel('Pixel Position')
ax4.set_ylabel('Intensity')
ax4.set_title('PSF Cross-Section')
ax4.legend()
ax4.set_xlim([center-50, center+50])
ax4.grid(True, alpha=0.3)

# DM shape
ax5 = fig.add_subplot(2, 3, 5)
dm_shape = system['dm'].coefs.reshape(-1)
n_act_side = int(np.sqrt(len(dm_shape))) if np.sqrt(len(dm_shape)) == int(np.sqrt(len(dm_shape))) else system['n_subaperture'] + 1
# Create a 2D representation
dm_2d = np.zeros((n_act_side, n_act_side))
dm_2d.flat[:len(dm_shape)] = dm_shape[:n_act_side**2] if len(dm_shape) >= n_act_side**2 else np.pad(dm_shape, (0, n_act_side**2 - len(dm_shape)))
im5 = ax5.imshow(dm_2d * 1e6, cmap='RdBu_r', origin='lower')
ax5.set_title('Final DM Shape')
ax5.set_xlabel('Actuator X')
ax5.set_ylabel('Actuator Y')
plt.colorbar(im5, ax=ax5, label='μm', shrink=0.8)

# Singular values
ax6 = fig.add_subplot(2, 3, 6)
ax6.semilogy(calibration['singular_values'], 'bo-', markersize=4)
ax6.axhline(y=calibration['singular_values'][0] * 1e-3, color='r', linestyle='--', 
            label=f'Threshold (10⁻³)')
ax6.set_xlabel('Mode Index')
ax6.set_ylabel('Singular Value')
ax6.set_title('Interaction Matrix Singular Values')
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ao_results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print quantitative metrics
print("\n" + "="*60)
print("QUANTITATIVE METRICS")
print("="*60)
print(f"Open-Loop Strehl:    {strehl_open:.2f}%")
print(f"Closed-Loop Strehl:  {results['strehl_history'][-1]:.2f}%")
print(f"Improvement Factor:  {results['strehl_history'][-1]/max(strehl_open, 0.01):.1f}x")
print(f"DM RMS:              {np.sqrt(np.mean(results['final_dm_commands']**2))*1e6:.3f} μm")
print("="*60)

## Convergence Analysis

The convergence behavior of the closed-loop AO system depends on several factors:

### Expected Behavior

1. **Initial Transient**: The first few iterations show rapid improvement as the integrator builds up the correction. The Strehl should increase quickly.

2. **Steady State**: After convergence, the Strehl should stabilize around a value determined by:
   - Fitting error (modes not corrected)
   - Temporal error (lag between measurement and correction)
   - Noise propagation

3. **Gain Effects**:
   - Too low ($g < 0.2$): Slow convergence, poor rejection of fast turbulence
   - Optimal ($g \approx 0.4-0.5$): Good balance of speed and stability
   - Too high ($g > 0.7$): Oscillations, potential instability

### Diagnostic Plots

We will examine:
1. **Strehl vs. Iteration**: Should show monotonic increase then plateau
2. **Residual Slopes RMS**: Should decrease as correction improves
3. **DM RMS**: Should increase as the mirror builds up correction

### Potential Issues

- **Oscillations**: Indicate gain too high or poor reconstructor conditioning
- **Slow convergence**: Gain too low or temporal bandwidth mismatch
- **Divergence**: Severe miscalibration or instability

In [ ]:
# =============================================================================
# Cell 11: Convergence Curve Plots
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

iterations = np.arange(1, len(results['strehl_history']) + 1)

# Plot 1: Strehl Ratio Evolution
axes[0].plot(iterations, results['strehl_history'], 'bo-', linewidth=2, markersize=6)
axes[0].axhline(y=results['strehl_history'][-1], color='g', linestyle='--', 
                alpha=0.7, label=f'Final: {results["strehl_history"][-1]:.1f}%')
axes[0].fill_between(iterations, 0, results['strehl_history'], alpha=0.2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Strehl Ratio [%]')
axes[0].set_title('Strehl Ratio Convergence')
axes[0].set_xlim([0, len(iterations) + 1])
axes[0].set_ylim([0, max(results['strehl_history']) * 1.1])
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Plot 2: Residual Slopes RMS
axes[1].semilogy(iterations, results['residual_slopes_history'], 'rs-', 
                  linewidth=2, markersize=6)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Residual Slopes RMS')
axes[1].set_title('Residual Wavefront Error')
axes[1].set_xlim([0, len(iterations) + 1])
axes[1].grid(True, alpha=0.3)

# Plot 3: DM RMS Command
axes[2].plot(iterations, np.array(results['dm_rms_history']) * 1e6, 'g^-', 
             linewidth=2, markersize=6)
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('DM RMS [μm]')
axes[2].set_title('DM Command Buildup')
axes[2].set_xlim([0, len(iterations) + 1])
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ao_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

# Convergence statistics
print("\nConvergence Statistics:")
print("-" * 40)
print(f"Initial Strehl:     {results['strehl_history'][0]:.2f}%")
print(f"Final Strehl:       {results['strehl_history'][-1]:.2f}%")
print(f"Convergence Rate:   {(results['strehl_history'][-1] - results['strehl_history'][0])/len(iterations):.2f}%/iter")
print(f"Iterations to 90%:  {np.argmax(results['strehl_history'] > 0.9*results['strehl_history'][-1]) + 1}")
print(f"Slope RMS Reduction: {results['residual_slopes_history'][0]/results['residual_slopes_history'][-1]:.1f}x")

## Error Analysis & Sensitivity

### Sources of Error in AO Systems

1. **Fitting Error**: The DM cannot correct spatial frequencies higher than its actuator spacing. For a continuous facesheet DM with actuator pitch $d$:
   $$\sigma_{\text{fit}}^2 \approx 0.28 \left(\frac{d}{r_0}\right)^{5/3}$$

2. **Temporal Error**: The finite loop bandwidth causes lag. For wind speed $v$ and loop delay $\tau$:
   $$\sigma_{\text{temp}}^2 \approx \left(\frac{v \tau}{r_0}\right)^{5/3}$$

3. **Measurement Noise**: Photon noise and read noise in the WFS propagate through the reconstructor:
   $$\sigma_{\text{noise}}^2 \propto \text{Tr}(\mathbf{R}^T \mathbf{R}) \cdot \sigma_s^2$$

4. **Aliasing Error**: High-frequency aberrations alias into low-frequency measurements.

### Sensitivity Analysis

We will examine how the reconstruction quality depends on:
- **Loop gain**: Trade-off between speed and stability
- **Number of modes**: More modes = better correction but more noise
- **Atmospheric conditions**: Stronger turbulence is harder to correct

### Regularization Effects

The SVD truncation threshold affects:
- **Too aggressive** (high threshold): Lose correction capability for some modes
- **Too permissive** (low threshold): Noise amplification in poorly-sensed modes

In [ ]:
# =============================================================================
# Cell 13: Error Map & Sensitivity Analysis
# =============================================================================

# Compute error metrics
final_psf = results['final_psf']
psf_error = np.abs(final_psf - psf_ref)
psf_error_normalized = psf_error / np.max(psf_ref)

# Create error visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Error map
im0 = axes[0, 0].imshow(np.log10(psf_error_normalized + 1e-10), cmap='viridis', origin='lower')
axes[0, 0].set_title('PSF Error Map (log scale)')
axes[0, 0].set_xlabel('Pixels')
axes[0, 0].set_ylabel('Pixels')
plt.colorbar(im0, ax=axes[0, 0], label='log₁₀(|Error|/max(Ref))')

# Radial profile comparison
center = psf_ref.shape[0] // 2
r = np.arange(0, center)
radial_ref = np.zeros(len(r))
radial_corrected = np.zeros(len(r))
radial_open = np.zeros(len(r))

for i, radius in enumerate(r):
    mask = (np.sqrt((np.arange(psf_ref.shape[0])[:, None] - center)**2 + 
                    (np.arange(psf_ref.shape[1])[None, :] - center)**2) >= radius) & \
           (np.sqrt((np.arange(psf_ref.shape[0])[:, None] - center)**2 + 
                    (np.arange(psf_ref.shape[1])[None, :] - center)**2) < radius + 1)
    if np.sum(mask) > 0:
        radial_ref[i] = np.mean(psf_ref[mask])
        radial_corrected[i] = np.mean(final_psf[mask])
        radial_open[i] = np.mean(psf_open_loop[mask])

axes[0, 1].semilogy(r, radial_open, 'r-', alpha=0.7, label='Open-Loop')
axes[0, 1].semilogy(r, radial_corrected, 'b-', linewidth=2, label='Closed-Loop')
axes[0, 1].semilogy(r, radial_ref, 'k--', label='Reference')
axes[0, 1].set_xlabel('Radius [pixels]')
axes[0, 1].set_ylabel('Azimuthally Averaged Intensity')
axes[0, 1].set_title('Radial PSF Profile')
axes[0, 1].legend()
axes[0, 1].set_xlim([0, 50])
axes[0, 1].grid(True, alpha=0.3)

# Encircled energy
ee_ref = np.cumsum(radial_ref * 2 * np.pi * r) / np.sum(psf_ref)
ee_corrected = np.cumsum(radial_corrected * 2 * np.pi * r) / np.sum(final_psf)
ee_open = np.cumsum(radial_open * 2 * np.pi * r) / np.sum(psf_open_loop)

axes[0, 2].plot(r, ee_open * 100, 'r-', alpha=0.7, label='Open-Loop')
axes[0, 2].plot(r, ee_corrected * 100, 'b-', linewidth=2, label='Closed-Loop')
axes[0, 2].plot(r, ee_ref * 100, 'k--', label='Reference')
axes[0, 2].axhline(y=50, color='gray', linestyle=':', alpha=0.5)
axes[0, 2].set_xlabel('Radius [pixels]')
axes[0, 2].set_ylabel('Encircled Energy [%]')
axes[0, 2].set_title('Encircled Energy')
axes[0, 2].legend()
axes[0, 2].set_xlim([0, 50])
axes[0, 2].set_ylim([0, 100])
axes[0, 2].grid(True, alpha=0.3)

# Sensitivity: Gain variation study
print("Running gain sensitivity study...")
gains = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
final_strehls = []

for g in gains:
    # Reset system
    system['dm'].coefs[:] = 0
    system['atm'].initializeAtmosphere(system['tel'])
    
    # Quick 10-iteration run
    strehl_temp = []
    for k in range(10):
        system['atm'].update()
        system['atm'] * system['ngs'] * system['tel'] * system['dm']
        slopes_meas = get_slopes_diffractive(system['wfs'], system['tel'], system['ngs']) - ref_slopes
        delta_cmd = np.matmul(calibration['reconstructor'], slopes_meas)
        system['dm'].coefs = system['dm'].coefs - g * delta_cmd
        system['atm'] * system['ngs'] * system['tel'] * system['dm'] * system['sci_cam']
        strehl_temp.append(compute_strehl(system['sci_cam'].frame, psf_ref))
    final_strehls.append(strehl_temp[-1])

axes[1, 0].bar(range(len(gains)), final_strehls, color='steelblue', edgecolor='black')
axes[1, 0].set_xticks(range(len(gains)))
axes[1, 0].set_xticklabels([f'{g:.1f}' for g in gains])
axes[1, 0].set_xlabel('Loop Gain')
axes[1, 0].set_ylabel('Final Strehl [%]')
axes[1, 0].set_title('Gain Sensitivity')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Mode contribution analysis
mode_contributions = np.abs(calibration['singular_values'])**2
mode_contributions = mode_contributions / np.sum(mode_contributions) * 100
cumulative = np.cumsum(mode_contributions)

axes[1, 1].bar(range(len(mode_contributions)), mode_contributions, 
               color='coral', edgecolor='black', alpha=0.7, label='Individual')
ax_twin = axes[1, 1].twinx()
ax_twin.plot(range(len(cumulative)), cumulative, 'b-', linewidth=2, label='Cumulative')
axes[1, 1].set_xlabel('Mode Index')
axes[1, 1].set_ylabel('Contribution [%]', color='coral')
ax_twin.set_ylabel('Cumulative [%]', color='blue')
axes[1, 1].set_title('Modal Contribution to Correction')
axes[1, 1].grid(True, alpha=0.3)

# Noise propagation coefficient
R = calibration['reconstructor']
noise_prop = np.sqrt(np.diag(R @ R.T))
axes[1, 2].plot(noise_prop, 'go-', markersize=3)
axes[1, 2].set_xlabel('Actuator Index')
axes[1, 2].set_ylabel('Noise Propagation Coefficient')
axes[1, 2].set_title('Noise Sensitivity per Actuator')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ao_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSensitivity Analysis Complete.")
print(f"Optimal gain from test: {gains[np.argmax(final_strehls)]:.1f}")

## Discussion & Key Takeaways

### Summary of Results

This tutorial demonstrated a complete adaptive optics simulation using the Shack-Hartmann wavefront sensor and closed-loop integral control. The key steps were:

1. **System Initialization**: Created a realistic 8m telescope with atmospheric turbulence, deformable mirror, and Shack-Hartmann WFS
2. **Calibration**: Measured the interaction matrix using push-pull and computed the reconstructor via regularized SVD
3. **Closed-Loop Control**: Implemented integral control to iteratively correct wavefront aberrations
4. **Performance Evaluation**: Achieved significant Strehl ratio improvement through AO correction

### Key Findings

1. **Inverse Problem Nature**: Wavefront reconstruction from slope measurements is inherently ill-posed. Regularization (SVD truncation) is essential to prevent noise amplification.

2. **Modal vs. Zonal**: Using a KL modal basis optimized for atmospheric statistics provides better noise rejection than direct zonal reconstruction.

3. **Control Dynamics**: The integral controller gain must balance convergence speed against stability. Typical optimal gains are 0.3-0.5.

4. **Performance Limits**: Even with perfect reconstruction, fitting error and temporal error limit achievable Strehl.

### Limitations

- **Single conjugate**: This system corrects only ground-layer turbulence; multi-conjugate AO would be needed for volume correction
- **Simplified noise model**: Real systems have photon noise, read noise, and background
- **No non-common path aberrations**: Real systems have static aberrations between WFS and science paths

### Extensions

- **Predictive control**: Use temporal prediction to reduce servo lag error
- **Optimal reconstruction**: Minimum variance estimators can outperform least-squares
- **Machine learning**: Neural networks can learn non-linear reconstructors

### Key References

1. Roddier, F. (1999). *Adaptive Optics in Astronomy*. Cambridge University Press.
2. Hardy, J. W. (1998). *Adaptive Optics for Astronomical Telescopes*. Oxford University Press.
3. Guyon, O. (2005). "Limits of Adaptive Optics for High-Contrast Imaging." *ApJ*, 629, 592.

In [ ]:
# =============================================================================
# Cell 15: Summary Metrics Table
# =============================================================================

# Compute all summary statistics
strehl_history = results['strehl_history']
initial_strehl = strehl_history[0]
final_strehl = strehl_history[-1]
mean_strehl = np.mean(strehl_history)
max_strehl = np.max(strehl_history)
min_strehl = np.min(strehl_history)
dm_rms = np.sqrt(np.mean(results['final_dm_commands']**2))
n_iterations = len(strehl_history)
improvement = final_strehl / max(strehl_open, 0.01)

# System parameters
n_subap = system['n_subaperture']
n_modes = system['n_modes']
n_actuators = system['dm'].nValidAct
n_slopes = len(ref_slopes)

# Print formatted summary table
print("\n" + "="*70)
print("                    ADAPTIVE OPTICS SIMULATION SUMMARY")
print("="*70)
print("\n┌─────────────────────────────────────────────────────────────────────┐")
print("│                        SYSTEM PARAMETERS                            │")
print("├─────────────────────────────────────────────────────────────────────┤")
print(f"│  Telescope Diameter:          {system['tel'].D:>10.1f} m                       │")
print(f"│  Fried Parameter (r0):        {system['atm'].r0:>10.2f} m                       │")
print(f"│  D/r0 Ratio:                  {system['tel'].D/system['atm'].r0:>10.1f}                          │")
print(f"│  Number of Subapertures:      {n_subap:>10d} x {n_subap}                      │")
print(f"│  Number of Valid Actuators:   {n_actuators:>10d}                          │")
print(f"│  Number of Slope Measurements:{n_slopes:>10d}                          │")
print(f"│  Number of Corrected Modes:   {n_modes:>10d}                          │")
print("└─────────────────────────────────────────────────────────────────────┘")
print("\n┌─────────────────────────────────────────────────────────────────────┐")
print("│                      PERFORMANCE METRICS                            │")
print("├─────────────────────────────────────────────────────────────────────┤")
print(f"│  Open-Loop Strehl:            {strehl_open:>10.2f} %                        │")
print(f"│  Initial Closed-Loop Strehl:  {initial_strehl:>10.2f} %                        │")
print(f"│  Final Closed-Loop Strehl:    {final_strehl:>10.2f} %                        │")
print(f"│  Mean Strehl:                 {mean_strehl:>10.2f} %                        │")
print(f"│  Maximum Strehl:              {max_strehl:>10.2f} %                        │")
print(f"│  Minimum Strehl:              {min_strehl:>10.2f} %                        │")
print(f"│  Improvement Factor:          {improvement:>10.1f} x                        │")
print("└─────────────────────────────────────────────────────────────────────┘")
print("\n┌─────────────────────────────────────────────────────────────────────┐")
print("│                      CONTROL PARAMETERS                             │")
print("├─────────────────────────────────────────────────────────────────────┤")
print(f"│  Loop Gain:                   {0.4:>10.2f}                          │")
print(f"│  Number of Iterations:        {n_iterations:>10d}                          │")
print(f"│  Final DM RMS:                {dm_rms*1e6:>10.4f} μm                      │")
print(f"│  Valid Modes (after SVD):     {calibration['n_valid_modes']:>10d}                          │")
print("└─────────────────────────────────────────────────────────────────────┘")
print("\n┌─────────────────────────────────────────────────────────────────────┐")
print("│                        OUTPUT FILES                                 │")
print("├─────────────────────────────────────────────────────────────────────┤")
print("│  ao_results_comparison.png    - PSF comparison plots                │")
print("│  ao_convergence.png           - Convergence curves                  │")
print("│  ao_error_analysis.png        - Error and sensitivity analysis      │")
print("└─────────────────────────────────────────────────────────────────────┘")
print("\n" + "="*70)
print("                    SIMULATION COMPLETED SUCCESSFULLY")
print("="*70)

## Conclusion

This tutorial presented a comprehensive implementation of adaptive optics wavefront sensing and correction using the Shack-Hartmann sensor. We addressed the fundamental **inverse problem** of reconstructing wavefront aberrations from local slope measurements and demonstrated closed-loop correction using integral control.

### Problem Recap
- **Forward Model**: Atmospheric turbulence → Wavefront aberration → SH-WFS slope measurements
- **Inverse Problem**: Given slopes, reconstruct wavefront and compute DM correction
- **Solution Method**: Modal decomposition + regularized least-squares + integral control

### Key Results
- Successfully implemented push-pull calibration to measure the interaction matrix
- Demonstrated SVD-based reconstruction with regularization to handle ill-conditioning
- Achieved significant Strehl ratio improvement through closed-loop AO correction
- Analyzed convergence behavior and sensitivity to control parameters

### Broader Impact
Adaptive optics enables ground-based telescopes to achieve near diffraction-limited imaging, revolutionizing astronomy by allowing detailed observations of exoplanets, stellar surfaces, and galactic nuclei. The same principles apply to ophthalmology (retinal imaging), laser communications, and microscopy.

The inverse problem framework presented here—measurement, reconstruction, and feedback control—is fundamental to many sensing and control applications beyond adaptive optics.